In [1]:
import requests
import time
import pandas as pd
from pathlib import Path

# CVR API Base URL
BASE_URL = "https://cvrapi.dk/api"

# Default params for the API
# IMPORTANT: Provide a User-Agent identifying your application to avoid being blocked
headers = {
    "User-Agent": "MSc_Project_Data_Fetcher - Educational Use"
}

In [ ]:
# List of companies to search for
companies = [
    "Genmab",
    "Novo Nordisk",
    "Lego",
]

results = []

print("Fetching data from CVR API...")
for company in companies:
    print(f"Searching for: {company}")
    params = {
        "country": "dk",
        "search": company
    }
    
    response = requests.get(BASE_URL, params=params, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        if "error" in data:
            print(f"  Error for {company}: {data['error']}")
        else:
            results.append(data)
            print(f"  Success: Found {data.get('name', 'Unknown')}, CVR: {data.get('vat', 'Unknown')}")
    else:
        print(f"  Failed with status code: {response.status_code}")
        
    # Be nice to the API - add a short delay
    time.sleep(1)

print(f"\nFetched data for {len(results)} companies.")

Fetching data from CVR API...
Searching for: Genmab
  Success: Found GENMAB A/S, CVR: 21023884
Searching for: Novo Nordisk
  Success: Found NOVO NORDISK A/S, CVR: 24256790
Searching for: Lego
  Success: Found LEGO Holding A/S, CVR: 28122454
Searching for: Carlsberg
  Success: Found CARLSBERG A/S, CVR: 61056416

Fetched data for 4 companies.


In [28]:
data_rows = []
index_list = []
keys = ['name', 'address', 'zipcode', 'city', 'startdate', 'enddate', 'employees']
columns = ['CompanyVat', 'CompanyName', 'UnitName', 'UnitAddress', 'UnitZipcode', 'UnitCity', 'UnitStartdate', 'UnitEnddate', 'UnitEmployees']

for res in results:
    for unit in res['productionunits']:
        pno = unit.get('pno', None)
        values = [res['vat'], res['name']] + [unit.get(key, None) for key in keys]
        
        data_rows.append(values)
        index_list.append(pno)

# Create DataFrame
df_units = pd.DataFrame(data_rows, columns=columns, index=index_list)
#df_units.set_index('pno', inplace=True)
df_units.index.name = 'pno'

display(df_units.head())

,CompanyVat,CompanyName,UnitName,UnitAddress,UnitZipcode,UnitCity,UnitStartdate,UnitEnddate,UnitEmployees
pno,,,,,,,,,
1004824743,21023884,GENMAB A/S,GENMAB A/S,Carl Jacobsens Vej 30,2500,Valby,11/06 - 1998,NaN,624
1029560494,21023884,GENMAB A/S,Genmab A/S,Baltorpvej 154,2750,Ballerup,18/02 - 2023,NaN,None
1017661031,24256790,NOVO NORDISK A/S,NOVO NORDISK A/S,Krogshøjvej 42,2880,Bagsværd,09/05 - 2012,30/06 - 2013,None
1017661341,24256790,NOVO NORDISK A/S,NOVO NORDISK A/S,Nybrovej 90,2820,Gentofte,09/05 - 2012,30/06 - 2013,None
1007675530,24256790,NOVO NORDISK A/S,Novo Nordisk A/S,Novo Alle 1,2880,Bagsværd,18/10 - 1999,18/10 - 1999,None


In [3]:
# Convert results to a pandas DataFrame and save to resources folder
if results:
    df_cvr = pd.DataFrame(results)
    
    # Ensure resources directory exists
    resources_dir = Path("../resources")
    resources_dir.mkdir(parents=True, exist_ok=True)
    
    output_path = resources_dir / "cvr_data.json"
    
    # Save as JSON (preserves nested dictionary structures better than CSV)
    df_cvr.to_json(output_path, orient="records", indent=4)
    print(f"Saved corporate data to {output_path}")
    
    # Also save a simplified CSV version for easier direct viewing
    cols_to_save = ['vat', 'name', 'address', 'zipcode', 'city', 'phone', 'email', 'employees']
    # Keep only available columns
    available_cols = [col for col in cols_to_save if col in df_cvr.columns]
    
    csv_out_path = resources_dir / "cvr_data.csv"
    df_cvr[available_cols].to_csv(csv_out_path, index=False)
    print(f"Saved simplified CSV to {csv_out_path}")
    
    display(df_cvr[available_cols].head())
else:
    print("No data to save.")

Saved corporate data to ../resources/cvr_data.json
Saved simplified CSV to ../resources/cvr_data.csv


,vat,name,address,zipcode,city,phone,email,employees
0,21023884,GENMAB A/S,Carl Jacobsens Vej 30,2500,Valby,70202728,NaN,624
1,24256790,NOVO NORDISK A/S,Novo Alle 1,2880,Bagsværd,44448888,NaN,30713
2,28122454,LEGO Holding A/S,Koldingvej 2,7190,Billund,75338833,kirkbi@kirkbi.com,4
3,61056416,CARLSBERG A/S,J.C. Jacobsens Gade 1,1799,København V,33273327,NaN,114


In [23]:
df_cvr.head()

,vat,name,address,zipcode,city,cityname,protected,phone,email,fax,...,industrydesc,companycode,companydesc,creditstartdate,creditbankrupt,creditstatus,owners,productionunits,t,version
0,21023884,GENMAB A/S,Carl Jacobsens Vej 30,2500,Valby,None,False,70202728,NaN,NaN,...,Forskning og eksperimentel udvikling inden for...,60,Aktieselskab,None,False,None,"[{'name': 'BLACKROCK, INC.'}]","[{'pno': 1004824743, 'main': True, 'name': 'GE...",100,6
1,24256790,NOVO NORDISK A/S,Novo Alle 1,2880,Bagsværd,None,False,44448888,NaN,NaN,...,Fremstilling af farmaceutiske præparater,60,Aktieselskab,None,False,None,None,"[{'pno': 1017661031, 'main': False, 'name': 'N...",100,6
2,28122454,LEGO Holding A/S,Koldingvej 2,7190,Billund,None,False,75338833,kirkbi@kirkbi.com,NaN,...,Leasing af intellektuelle ejendomsrettigheder ...,60,Aktieselskab,None,False,None,None,"[{'pno': 1010823230, 'main': True, 'name': 'LE...",100,6
3,61056416,CARLSBERG A/S,J.C. Jacobsens Gade 1,1799,København V,None,False,33273327,NaN,33274709,...,Forskning og eksperimentel udvikling inden for...,60,Aktieselskab,None,False,None,[{'name': 'Massachusetts Financial Services Co...,"[{'pno': 1009961212, 'main': False, 'name': 'C...",100,6
